# 017 — RAGAS: End-to-End RAG Evaluation
**Evaluation + Observability** | Automated metrics for your RAG pipeline

RAGAS metrics:
| Metric | Question |
|---|---|
| **Faithfulness** | Is the answer supported by the retrieved context? |
| **Answer Relevance** | Is the answer actually answering the question? |
| **Context Recall** | Was the right information retrieved? |
| **Context Precision** | Were the retrieved docs relevant (no noise)? |


In [1]:
!pip install -q ragas

In [2]:
import os, warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', message=".*LangChain.*")
warnings.filterwarnings('ignore', message=".*allowed_objects.*")
warnings.filterwarnings('ignore', module="langchain.*")
warnings.filterwarnings('ignore', module="langgraph.*")

from dotenv import load_dotenv
load_dotenv(os.path.join(os.getcwd(), '..', '.env'))
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
assert GROQ_API_KEY, "GROQ_API_KEY not found — check ../.env"
print("✓ API key loaded")


✓ API key loaded


In [3]:
import os, time
os.environ["ANONYMIZED_TELEMETRY"] = "False"  # silence ChromaDB telemetry

from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0, max_retries=2)
llm_smart = ChatGroq(model="llama-3.3-70b-versatile", temperature=0, max_retries=2)
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    cache_folder=os.path.join(os.getcwd(), '..', 'datasets', 'models'),
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
print(f"✓ llm       : {llm.model_name}")
print(f"✓ llm_smart : {llm_smart.model_name}")
print(f"✓ embeddings: all-MiniLM-L6-v2 (384-dim, normalized cosine)")


✓ llm       : llama-3.1-8b-instant
✓ llm_smart : llama-3.3-70b-versatile
✓ embeddings: all-MiniLM-L6-v2 (384-dim, normalized cosine)


In [4]:
import os

DS_DIR = os.path.join(os.getcwd(), '..', 'datasets')
os.makedirs(DS_DIR, exist_ok=True)

def load_doc(fname):
    path = os.path.join(DS_DIR, fname)
    if os.path.exists(path):
        return open(path, encoding='utf-8').read()
    raise FileNotFoundError(
        f"Missing: {path}\n"
        "Run  python create_datasets.py  from the project root."
    )

ai_text  = load_doc("ai.txt")
ml_text  = load_doc("machine_learning.txt")
nlp_text = load_doc("nlp.txt")
dl_text  = load_doc("deep_learning.txt")
rag_text = load_doc("rag.txt")
all_text = "\n\n".join([ai_text, ml_text, nlp_text, dl_text, rag_text])

print(f"✓ 5 dataset files loaded")
print(f"  {'File':<20} {'Chars':>8}  {'~Words':>7}")
print(f"  {'-'*40}")
for name, txt in [('ai.txt', ai_text), ('machine_learning.txt', ml_text),
                  ('nlp.txt', nlp_text), ('deep_learning.txt', dl_text),
                  ('rag.txt', rag_text)]:
    print(f"  {name:<20} {len(txt):>8,}  {len(txt.split()):>7,}")
print(f"  {'-'*40}")
print(f"  {'TOTAL':<20} {len(all_text):>8,}  {len(all_text.split()):>7,}")


✓ 5 dataset files loaded
  File                    Chars   ~Words
  ----------------------------------------
  ai.txt                  6,221      943
  machine_learning.txt    5,953      870
  nlp.txt                 5,160      693
  deep_learning.txt       5,114      686
  rag.txt                 5,146      715
  ----------------------------------------
  TOTAL                  27,602    3,907


In [5]:
# ── Build test RAG pipeline ───────────────────────────────────────────────
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=50)
docs = [Document(page_content=c, metadata={"source": "wiki"})
        for text in [ai_text, ml_text, dl_text]
        for c in splitter.split_text(text[:15000])]
vectorstore = FAISS.from_documents(docs, embeddings)
retriever   = vectorstore.as_retriever(search_kwargs={"k": 4})

def run_rag(question: str):
    retrieved = retriever.invoke(question)
    ctx = "\n\n".join(d.page_content for d in retrieved)
    chain = (
        ChatPromptTemplate.from_template(
            "Answer using only context.\nContext:\n{ctx}\nQ: {q}\nA:"
        ) | llm | StrOutputParser()
    )
    return {
        "answer":   chain.invoke({"ctx": ctx[:3000], "q": question}),
        "contexts": [d.page_content for d in retrieved],
    }

print(f"✓ RAG pipeline over {len(docs)} docs")


✓ RAG pipeline over 47 docs


In [6]:
# ── Evaluation dataset (3 samples — keeps runtime under 2 min) ────────────
import time

eval_questions = [
    "What is the definition of machine learning?",
    "How does backpropagation work in neural networks?",
    "What are the main types of machine learning?",
]
ground_truths = [
    "Machine learning is a branch of AI that enables systems to learn from data without explicit programming.",
    "Backpropagation computes gradients of the loss using the chain rule and propagates them backwards through the network.",
    "The main types of machine learning are supervised learning, unsupervised learning, and reinforcement learning.",
]

print("Running RAG on 3 evaluation questions...")
eval_data = {"question": [], "answer": [], "contexts": [], "ground_truth": []}
t0 = time.time()
for q, gt in zip(eval_questions, ground_truths):
    result = run_rag(q)
    eval_data["question"].append(q)
    eval_data["answer"].append(result["answer"])
    eval_data["contexts"].append(result["contexts"])
    eval_data["ground_truth"].append(gt)
    print(f"  ✓ [{len(eval_data['question'])}/3] {q[:55]}")

print(f"\n✓ Eval dataset ready in {time.time()-t0:.1f}s")
print(f"  Sample answer: {eval_data['answer'][0][:150]}...")


Running RAG on 3 evaluation questions...


  ✓ [1/3] What is the definition of machine learning?
  ✓ [2/3] How does backpropagation work in neural networks?


  ✓ [3/3] What are the main types of machine learning?

✓ Eval dataset ready in 0.9s
  Sample answer: Machine learning is a method of data analysis that automates analytical model building. It is based on the idea that systems can learn from data, iden...


In [7]:
# ── RAGAS metrics — faithfulness only (most reliable on free-tier Groq) ──
# RAGAS 0.4.x renames: question→user_input, answer→response,
#                       contexts→retrieved_contexts, ground_truth→reference
# answer_relevancy is slow (generates synthetic questions); use faithfulness here.
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
import time

ragas_llm   = LangchainLLMWrapper(llm)   # fast model to stay within rate limits
ragas_embed = LangchainEmbeddingsWrapper(embeddings)

dataset = Dataset.from_dict(eval_data)
print(f"Running RAGAS faithfulness on {len(dataset)} samples...")
print("(faithfulness checks that every claim in the answer is backed by context)\n")

t0 = time.time()
results = None
try:
    results = evaluate(
        dataset=dataset,
        metrics=[faithfulness],
        llm=ragas_llm,
        embeddings=ragas_embed,
    )
    elapsed = time.time() - t0
    print(f"✓ Evaluation complete in {elapsed:.1f}s")
    # results['faithfulness'] can be a float or list depending on ragas version
    raw = results['faithfulness']
    score = sum(raw)/len(raw) if isinstance(raw, list) else float(raw)
    print(f"  Aggregate faithfulness: {score:.3f}")
except Exception as e:
    print(f"RAGAS error ({type(e).__name__}): {e}")


Running RAGAS faithfulness on 3 samples...
(faithfulness checks that every claim in the answer is backed by context)



Evaluating:   0%|          | 0/3 [00:00<?, ?it/s]

✓ Evaluation complete in 150.9s
  Aggregate faithfulness: 0.667


In [8]:
# ── Per-sample breakdown ─────────────────────────────────────────────────
# RAGAS 0.4.x to_pandas() columns: user_input, retrieved_contexts, response,
# reference, <metric_name>
import pandas as pd

if results is None:
    print("No results to display — evaluation failed above.")
else:
    df = results.to_pandas()
    print("Available columns:", df.columns.tolist())

    # Identify metric columns (float dtype, not the input fields)
    input_cols  = {"user_input", "retrieved_contexts", "response", "reference",
                   "question", "answer", "contexts", "ground_truth"}
    metric_cols = [c for c in df.columns if c not in input_cols]

    # Use whichever question column exists
    q_col = next((c for c in ["user_input", "question"] if c in df.columns), None)

    pd.set_option('display.max_colwidth', 55)
    pd.set_option('display.float_format', '{:.3f}'.format)

    display_cols = ([q_col] if q_col else []) + metric_cols
    print("\nPer-sample scores:")
    print(df[display_cols].to_string(index=False))

    print("\nAggregate scores (mean ± std):")
    thresholds = {"faithfulness": 0.85, "answer_relevancy": 0.80,
                  "context_recall": 0.75, "context_precision": 0.70}
    for col in metric_cols:
        vals  = df[col].dropna()
        score = vals.mean() if len(vals) else float("nan")
        std   = vals.std()  if len(vals) > 1 else 0.0
        thr   = thresholds.get(col, 0.75)
        icon  = "✓ GOOD" if score >= thr else "⚠ NEEDS WORK"
        print(f"  {col:<25}: {score:.3f} ± {std:.3f}   {icon}  (threshold ≥ {thr})")


Available columns: ['user_input', 'retrieved_contexts', 'response', 'reference', 'faithfulness']

Per-sample scores:
                                       user_input  faithfulness
      What is the definition of machine learning?         1.000
How does backpropagation work in neural networks?         1.000
     What are the main types of machine learning?         0.000

Aggregate scores (mean ± std):
  faithfulness             : 0.667 ± 0.577   ⚠ NEEDS WORK  (threshold ≥ 0.85)


## Key Takeaways
| Score | Meaning | Typical good threshold |
|---|---|---|
| Faithfulness | No hallucination | > 0.85 |
| Answer Relevancy | On-topic | > 0.80 |
| Context Recall | Right docs retrieved | > 0.75 |
| Context Precision | No noisy docs | > 0.70 |

- Run RAGAS on every RAG pipeline change — treat it like a unit test suite
- Low **faithfulness** → LLM is hallucinating; add stronger system prompt constraints
- Low **context recall** → retrieval is missing relevant docs; improve chunking/embedding
